# Intelligent Customer Support Chatbot for Real-Time Issue Resolution

## Project Overview
This notebook implements an AI-powered customer support chatbot that:
- Uses fine-tuned LLMs for domain-specific queries
- Implements RAG (Retrieval Augmented Generation) for knowledge base retrieval
- Provides real-time support with automatic escalation
- Includes a user-friendly interface
- Evaluates performance on real-world queries

## 1. Environment Setup and Dependencies

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q langchain langchain-community langchain-openai
!pip install -q chromadb sentence-transformers
!pip install -q gradio
!pip install -q openai tiktoken
!pip install -q faiss-cpu
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# Import libraries
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Transformers and model training
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# LangChain for RAG
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma, FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline

# UI and evaluation
import gradio as gr
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Data Collection and Preprocessing

We'll create sample customer support data and knowledge base documents. In production, replace this with your actual data.

In [ ]:
# Sample customer support conversations for fine-tuning
customer_support_data = [
    {
        "query": "How do I reset my password?",
        "response": "To reset your password, please follow these steps: 1) Go to the login page, 2) Click on 'Forgot Password', 3) Enter your registered email address, 4) Check your email for a password reset link, 5) Click the link and create a new password. If you don't receive the email within 5 minutes, please check your spam folder."
    },
    {
        "query": "My order hasn't arrived yet. What should I do?",
        "response": "I understand your concern about the delayed order. Please provide your order number so I can track it for you. Typically, orders arrive within 3-5 business days. If it's been longer than that, I'll escalate this to our shipping department immediately."
    },
    {
        "query": "How can I cancel my subscription?",
        "response": "To cancel your subscription: 1) Log into your account, 2) Go to 'Settings' > 'Subscription', 3) Click on 'Cancel Subscription', 4) Follow the prompts to confirm. Your subscription will remain active until the end of the current billing period. Would you like me to help with any specific issues before canceling?"
    },
    {
        "query": "What are your refund policies?",
        "response": "Our refund policy allows returns within 30 days of purchase for most items. The product must be unused and in original packaging. Digital products and personalized items are non-refundable. To initiate a refund, go to 'My Orders', select the item, and click 'Request Refund'. Refunds are processed within 5-7 business days."
    },
    {
        "query": "The app keeps crashing on my phone",
        "response": "I'm sorry to hear about the app crashes. Let's troubleshoot: 1) Make sure you have the latest version of the app, 2) Try clearing the app cache in your phone settings, 3) Restart your device, 4) If the issue persists, uninstall and reinstall the app. What type of phone are you using and what version of the app do you have?"
    },
    {
        "query": "How do I update my billing information?",
        "response": "To update your billing information: 1) Log into your account, 2) Navigate to 'Account Settings', 3) Select 'Payment Methods', 4) Click 'Edit' next to your current payment method or 'Add New Payment Method', 5) Enter your new billing details and save. Your information is encrypted and secure."
    },
    {
        "query": "Can I change my delivery address?",
        "response": "Yes, you can change your delivery address if your order hasn't been shipped yet. Go to 'My Orders', find your order, and click 'Edit Address'. If the order has already been shipped, please contact our support team immediately and we'll try to redirect the package or arrange a redelivery."
    },
    {
        "query": "Do you offer student discounts?",
        "response": "Yes! We offer a 20% student discount on most products. To verify your student status, you'll need to: 1) Create an account or log in, 2) Go to 'Discounts & Offers', 3) Click 'Verify Student Status', 4) Upload a valid student ID or use your .edu email address. The discount will be automatically applied to eligible purchases."
    },
    {
        "query": "I was charged twice for the same order",
        "response": "I apologize for the double charge. This is a serious issue that needs immediate attention. Please provide your order number and I'll escalate this to our billing department right away. You should see the duplicate charge reversed within 3-5 business days. I'll also send you a confirmation email once it's resolved."
    },
    {
        "query": "How do I track my order?",
        "response": "To track your order: 1) Log into your account, 2) Go to 'My Orders', 3) Click on the specific order, 4) You'll see the tracking number and current status. You can also click the tracking number to see detailed shipping information. You should have also received a tracking email when your order was shipped."
    }
]

# Save to JSON file
os.makedirs('data', exist_ok=True)
with open('data/customer_support_conversations.json', 'w') as f:
    json.dump(customer_support_data, f, indent=2)

print(f"Created {len(customer_support_data)} sample conversations")

In [ ]:
# Create knowledge base documents
knowledge_base_docs = [
    {
        "title": "Password Reset Procedure",
        "content": "To reset your password: Navigate to the login page, click 'Forgot Password', enter your registered email, check your inbox for a reset link (valid for 24 hours), create a new password with at least 8 characters including uppercase, lowercase, numbers, and special characters. If you don't receive the email, check spam folder or contact support."
    },
    {
        "title": "Shipping and Delivery Information",
        "content": "Standard shipping takes 3-5 business days. Express shipping is 1-2 business days. International orders take 7-14 business days. Free shipping on orders over $50. Tracking information is sent via email once order is shipped. Shipping addresses can be modified before shipment. PO Box delivery available for standard shipping only."
    },
    {
        "title": "Refund and Return Policy",
        "content": "30-day return window for most products. Items must be unused, in original packaging with tags attached. Digital products, personalized items, and opened software are non-refundable. Refund processing takes 5-7 business days after receiving the returned item. Original shipping costs are non-refundable unless the return is due to our error. Return shipping is customer's responsibility unless product is defective."
    },
    {
        "title": "Subscription Management",
        "content": "Subscriptions can be modified or cancelled anytime from Account Settings. Cancellations take effect at the end of the current billing period. No refunds for partial months. Subscription tiers: Basic ($9.99/month), Premium ($19.99/month), Enterprise ($49.99/month). Annual subscriptions save 20%. Automatic renewal can be disabled in settings."
    },
    {
        "title": "Technical Support - Mobile App",
        "content": "Common app issues: Update to latest version from app store, clear app cache in device settings, ensure stable internet connection, restart device, reinstall app if problems persist. Minimum requirements: iOS 13+ or Android 8+, 2GB RAM, 100MB storage. Supported devices: iPhone 6s and newer, Android devices from 2018 onwards. For persistent issues, contact technical support with device model and app version."
    },
    {
        "title": "Account Security",
        "content": "Enable two-factor authentication (2FA) in security settings for enhanced protection. Strong passwords required: minimum 8 characters with mixed case, numbers, and symbols. Account lockout after 5 failed login attempts. Session timeout after 30 minutes of inactivity. Suspicious activity notifications sent to registered email. Regular security audits recommended. Never share passwords or 2FA codes."
    },
    {
        "title": "Payment Methods",
        "content": "Accepted payment methods: Visa, Mastercard, American Express, Discover, PayPal, Apple Pay, Google Pay. All transactions encrypted with SSL. Card verification (CVV) required. Billing address must match card registration. Multiple payment methods can be saved. Primary payment method used for subscriptions. Payment failures trigger email notification. Update payment info in Account Settings > Payment Methods."
    },
    {
        "title": "Student Discount Program",
        "content": "20% discount for verified students on annual subscriptions and most products. Verification required through valid .edu email or student ID upload. Discount valid while student status is active. Re-verification required annually. Cannot be combined with other promotional offers. Applied automatically at checkout after verification. Available to high school, college, and university students worldwide."
    },
    {
        "title": "Billing Issues and Disputes",
        "content": "Double charges are investigated within 24 hours. Temporary authorization holds may appear as duplicate charges and typically clear within 3-5 business days. Disputed charges should be reported immediately through support ticket. Full transaction history available in Account > Billing History. Invoice copies can be downloaded. For billing emergencies, use priority support channel. Refunds processed to original payment method."
    },
    {
        "title": "Order Tracking",
        "content": "Track orders through My Orders section or tracking link in shipment email. Tracking updates every 24 hours. Status meanings: Processing (order confirmed), Shipped (in transit), Out for Delivery (arriving today), Delivered (completed). Tracking numbers provided by shipping carrier. Delivery signature may be required for high-value items. Contact support if tracking shows no updates for 48 hours."
    }
]

# Save knowledge base
with open('data/knowledge_base.json', 'w') as f:
    json.dump(knowledge_base_docs, f, indent=2)

print(f"Created knowledge base with {len(knowledge_base_docs)} documents")

In [ ]:
# Prepare data for fine-tuning
def prepare_training_data(conversations: List[Dict]) -> Dataset:
    """
    Convert conversations to training format for instruction fine-tuning
    """
    formatted_data = []
    
    for conv in conversations:
        # Format as instruction-following task
        instruction = f"""You are a helpful customer support assistant. Answer the following customer query professionally and accurately.

Customer Query: {conv['query']}

Assistant:"""
        
        formatted_data.append({
            "text": instruction,
            "response": conv['response']
        })
    
    return Dataset.from_list(formatted_data)

# Load and prepare dataset
with open('data/customer_support_conversations.json', 'r') as f:
    conversations = json.load(f)

train_dataset = prepare_training_data(conversations)
print(f"Training dataset size: {len(train_dataset)}")
print("\nSample training example:")
print(train_dataset[0])

## 3. LLM Fine-Tuning with LoRA

We'll fine-tune a language model using LoRA (Low-Rank Adaptation) for parameter-efficient training.

In [ ]:
# Model configuration
MODEL_NAME = "microsoft/phi-2"  # Using a smaller model for demonstration
# Alternative models: "mistralai/Mistral-7B-v0.1", "meta-llama/Llama-2-7b-hf"

# Quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded successfully!")

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "dense"],  # Adjust based on model architecture
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Prepare model for training
base_model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(base_model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

In [ ]:
# Tokenize dataset
def tokenize_function(examples):
    combined_text = [f"{text} {response}" for text, response in zip(examples['text'], examples['response'])]
    tokenized = tokenizer(
        combined_text,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

print("Dataset tokenized successfully!")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=10,
    optim="paged_adamw_8bit",
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("Trainer initialized. Ready to fine-tune!")

In [ ]:
# Fine-tune the model
print("Starting fine-tuning...")
trainer.train()

# Save the fine-tuned model
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

print("Fine-tuning complete! Model saved to ./fine_tuned_model")

## 4. RAG Implementation

Implement Retrieval Augmented Generation to fetch relevant information from the knowledge base.

In [ ]:
# Load knowledge base
with open('data/knowledge_base.json', 'r') as f:
    kb_docs = json.load(f)

# Create document texts
documents = [f"Title: {doc['title']}\n\nContent: {doc['content']}" for doc in kb_docs]

print(f"Loaded {len(documents)} knowledge base documents")

In [ ]:
# Text splitting for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

split_docs = text_splitter.create_documents(documents)
print(f"Split into {len(split_docs)} chunks")

In [ ]:
# Create embeddings and vector store
print("Creating embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings
)

# Save vector store
vectorstore.save_local("./vector_store")
print("Vector store created and saved!")

In [ ]:
# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Retrieve top 3 relevant documents
)

# Test retrieval
test_query = "How do I reset my password?"
retrieved_docs = retriever.get_relevant_documents(test_query)

print(f"Test query: {test_query}\n")
print(f"Retrieved {len(retrieved_docs)} relevant documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content[:200] + "...")

## 5. Chatbot Agent with Escalation Logic

Build the intelligent agent that combines the fine-tuned model with RAG and includes escalation capabilities.

In [ ]:
class CustomerSupportAgent:
    """
    Intelligent customer support agent with RAG and escalation logic
    """
    
    def __init__(self, model, tokenizer, retriever, escalation_keywords=None):
        self.model = model
        self.tokenizer = tokenizer
        self.retriever = retriever
        self.conversation_history = []
        
        # Keywords that trigger escalation
        self.escalation_keywords = escalation_keywords or [
            "urgent", "emergency", "legal", "lawsuit", "attorney",
            "manager", "supervisor", "unacceptable", "fraud",
            "charged twice", "double charge", "billing error",
            "never received", "still waiting", "angry", "frustrated"
        ]
        
        # Track escalation flag
        self.escalated = False
    
    def should_escalate(self, query: str) -> bool:
        """
        Determine if the query should be escalated to human support
        """
        query_lower = query.lower()
        
        # Check for escalation keywords
        for keyword in self.escalation_keywords:
            if keyword in query_lower:
                return True
        
        # Check for multiple failed attempts (conversation history)
        if len(self.conversation_history) > 5:
            return True
        
        return False
    
    def retrieve_context(self, query: str, k: int = 3) -> str:
        """
        Retrieve relevant context from knowledge base
        """
        retrieved_docs = self.retriever.get_relevant_documents(query)
        context = "\n\n".join([doc.page_content for doc in retrieved_docs[:k]])
        return context
    
    def generate_response(self, query: str, context: str) -> str:
        """
        Generate response using fine-tuned model with retrieved context
        """
        prompt = f"""You are a helpful customer support assistant. Use the following context to answer the customer's query.

Context:
{context}

Customer Query: {query}

Assistant: """
        
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the assistant's response
        response = response.split("Assistant:")[-1].strip()
        
        return response
    
    def chat(self, query: str) -> Dict:
        """
        Main chat interface
        """
        # Check for escalation
        if self.should_escalate(query):
            self.escalated = True
            return {
                "response": "I understand this is an important issue. I'm escalating your case to our specialized support team who will contact you within 1 hour. Your ticket number is #" + str(np.random.randint(10000, 99999)) + ". Is there anything else I can help you with in the meantime?",
                "escalated": True,
                "context_used": None
            }
        
        # Retrieve relevant context
        context = self.retrieve_context(query)
        
        # Generate response
        response = self.generate_response(query, context)
        
        # Add to conversation history
        self.conversation_history.append({
            "query": query,
            "response": response,
            "timestamp": datetime.now().isoformat()
        })
        
        return {
            "response": response,
            "escalated": False,
            "context_used": context
        }
    
    def reset_conversation(self):
        """
        Reset conversation history
        """
        self.conversation_history = []
        self.escalated = False

print("CustomerSupportAgent class defined!")

In [ ]:
# Initialize the customer support agent
agent = CustomerSupportAgent(
    model=model,
    tokenizer=tokenizer,
    retriever=retriever
)

print("Customer Support Agent initialized!")

In [ ]:
# Test the agent
test_queries = [
    "How do I reset my password?",
    "I was charged twice for my order!",
    "What's your refund policy?"
]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    
    result = agent.chat(query)
    
    print(f"\nResponse: {result['response']}")
    print(f"Escalated: {result['escalated']}")
    
    if result['context_used']:
        print(f"\nContext Used:\n{result['context_used'][:200]}...")

## 6. User Interface with Gradio

Create an interactive web interface for the chatbot.

In [ ]:
# Create Gradio interface
def chatbot_interface(message, history):
    """
    Gradio chatbot function
    """
    result = agent.chat(message)
    
    response = result['response']
    
    # Add escalation notice if applicable
    if result['escalated']:
        response = "🚨 ESCALATED TO HUMAN SUPPORT 🚨\n\n" + response
    
    return response

# Create the Gradio interface
demo = gr.ChatInterface(
    fn=chatbot_interface,
    title="🤖 Intelligent Customer Support Chatbot",
    description="""Ask me anything about our products and services! I can help with:
    - Account management (password reset, billing, subscriptions)
    - Order tracking and shipping
    - Returns and refunds
    - Technical support
    - General inquiries
    
    If I can't help, I'll escalate your issue to our human support team.""",
    examples=[
        "How do I reset my password?",
        "Where is my order?",
        "What's your refund policy?",
        "The app keeps crashing",
        "Do you offer student discounts?",
    ],
    theme="soft",
    retry_btn="🔄 Retry",
    undo_btn="↩️ Undo",
    clear_btn="🗑️ Clear",
)

print("Gradio interface created!")

In [ ]:
# Launch the interface
# Note: In Jupyter, this will create an inline interface
# Set share=True to create a public link

demo.launch(
    share=False,  # Set to True to create a public link
    debug=True,
    server_port=7860
)

## 7. Evaluation and Testing

Evaluate the chatbot's performance using various metrics.

In [ ]:
# Create test dataset with ground truth responses
test_cases = [
    {
        "query": "How do I reset my password?",
        "expected_keywords": ["login", "email", "forgot password", "reset link"],
        "should_escalate": False
    },
    {
        "query": "I was charged twice for the same order!",
        "expected_keywords": ["billing", "charge", "refund"],
        "should_escalate": True
    },
    {
        "query": "What is your refund policy?",
        "expected_keywords": ["30 days", "return", "refund"],
        "should_escalate": False
    },
    {
        "query": "The app keeps crashing on my phone",
        "expected_keywords": ["update", "cache", "reinstall"],
        "should_escalate": False
    },
    {
        "query": "I need to speak to a manager immediately!",
        "expected_keywords": ["escalate", "support", "team"],
        "should_escalate": True
    },
]

print(f"Created {len(test_cases)} test cases")

In [ ]:
# Run evaluation
def evaluate_chatbot(agent, test_cases):
    """
    Evaluate chatbot performance on test cases
    """
    results = []
    
    for i, test_case in enumerate(test_cases):
        # Reset agent for each test
        agent.reset_conversation()
        
        query = test_case['query']
        expected_keywords = test_case['expected_keywords']
        should_escalate = test_case['should_escalate']
        
        # Get response
        result = agent.chat(query)
        response = result['response'].lower()
        escalated = result['escalated']
        
        # Check keyword relevance
        keywords_found = sum(1 for kw in expected_keywords if kw.lower() in response)
        keyword_score = keywords_found / len(expected_keywords)
        
        # Check escalation accuracy
        escalation_correct = escalated == should_escalate
        
        results.append({
            'test_id': i + 1,
            'query': query,
            'response': result['response'][:100] + '...',
            'keyword_score': keyword_score,
            'escalation_correct': escalation_correct,
            'escalated': escalated,
            'should_escalate': should_escalate
        })
    
    return pd.DataFrame(results)

# Run evaluation
print("Running evaluation...")
eval_results = evaluate_chatbot(agent, test_cases)

print("\n" + "="*100)
print("EVALUATION RESULTS")
print("="*100)
print(eval_results.to_string())

# Calculate overall metrics
avg_keyword_score = eval_results['keyword_score'].mean()
escalation_accuracy = eval_results['escalation_correct'].mean()

print("\n" + "="*100)
print("SUMMARY METRICS")
print("="*100)
print(f"Average Keyword Relevance Score: {avg_keyword_score:.2%}")
print(f"Escalation Accuracy: {escalation_accuracy:.2%}")

In [ ]:
# Visualize evaluation results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Keyword Relevance Scores
axes[0].bar(eval_results['test_id'], eval_results['keyword_score'], color='steelblue')
axes[0].set_xlabel('Test Case ID')
axes[0].set_ylabel('Keyword Relevance Score')
axes[0].set_title('Keyword Relevance by Test Case')
axes[0].set_ylim([0, 1.1])
axes[0].axhline(y=avg_keyword_score, color='r', linestyle='--', label=f'Average: {avg_keyword_score:.2f}')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Escalation Accuracy
escalation_counts = eval_results['escalation_correct'].value_counts()
colors = ['#66c2a5', '#fc8d62']
axes[1].pie(
    escalation_counts.values,
    labels=['Correct', 'Incorrect'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
axes[1].set_title('Escalation Decision Accuracy')

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nEvaluation charts saved as 'evaluation_results.png'")

In [ ]:
# Response time analysis
import time

def measure_response_time(agent, queries, num_runs=3):
    """
    Measure average response time
    """
    times = []
    
    for query in queries:
        run_times = []
        for _ in range(num_runs):
            agent.reset_conversation()
            start_time = time.time()
            agent.chat(query)
            end_time = time.time()
            run_times.append(end_time - start_time)
        
        times.append({
            'query': query[:50] + '...',
            'avg_time': np.mean(run_times),
            'std_time': np.std(run_times)
        })
    
    return pd.DataFrame(times)

# Test queries
test_queries = [tc['query'] for tc in test_cases[:3]]

print("Measuring response times...")
time_results = measure_response_time(agent, test_queries)

print("\n" + "="*80)
print("RESPONSE TIME ANALYSIS")
print("="*80)
print(time_results.to_string())
print(f"\nOverall Average Response Time: {time_results['avg_time'].mean():.2f} seconds")

## 8. Advanced Features

Additional functionality for production deployment.

In [ ]:
# Conversation analytics
class ConversationAnalytics:
    """
    Track and analyze conversation metrics
    """
    
    def __init__(self):
        self.conversations = []
    
    def log_conversation(self, conversation_history, escalated, resolved):
        self.conversations.append({
            'timestamp': datetime.now(),
            'num_turns': len(conversation_history),
            'escalated': escalated,
            'resolved': resolved,
            'queries': [turn['query'] for turn in conversation_history]
        })
    
    def get_metrics(self):
        if not self.conversations:
            return {}
        
        df = pd.DataFrame(self.conversations)
        
        return {
            'total_conversations': len(df),
            'escalation_rate': df['escalated'].mean(),
            'resolution_rate': df['resolved'].mean(),
            'avg_conversation_length': df['num_turns'].mean(),
            'conversations_per_day': len(df) / max((df['timestamp'].max() - df['timestamp'].min()).days, 1)
        }
    
    def plot_metrics(self):
        if not self.conversations:
            print("No conversations to plot")
            return
        
        df = pd.DataFrame(self.conversations)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Escalation rate over time
        df['date'] = df['timestamp'].dt.date
        daily_escalation = df.groupby('date')['escalated'].mean()
        axes[0, 0].plot(daily_escalation.index, daily_escalation.values, marker='o')
        axes[0, 0].set_title('Daily Escalation Rate')
        axes[0, 0].set_ylabel('Escalation Rate')
        axes[0, 0].grid(True, alpha=0.3)
        
        # Conversation length distribution
        axes[0, 1].hist(df['num_turns'], bins=20, color='steelblue', edgecolor='black')
        axes[0, 1].set_title('Conversation Length Distribution')
        axes[0, 1].set_xlabel('Number of Turns')
        axes[0, 1].set_ylabel('Frequency')
        
        # Resolution vs Escalation
        resolution_data = df.groupby('escalated')['resolved'].mean()
        axes[1, 0].bar(['Not Escalated', 'Escalated'], resolution_data.values, color=['green', 'orange'])
        axes[1, 0].set_title('Resolution Rate by Escalation')
        axes[1, 0].set_ylabel('Resolution Rate')
        
        # Daily conversation volume
        daily_volume = df.groupby('date').size()
        axes[1, 1].bar(daily_volume.index, daily_volume.values, color='coral')
        axes[1, 1].set_title('Daily Conversation Volume')
        axes[1, 1].set_ylabel('Number of Conversations')
        
        plt.tight_layout()
        plt.savefig('conversation_analytics.png', dpi=300, bbox_inches='tight')
        plt.show()

analytics = ConversationAnalytics()
print("ConversationAnalytics initialized!")

In [ ]:
# Sentiment analysis for customer queries
from transformers import pipeline

# Load sentiment analysis model
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

def analyze_sentiment(text):
    """
    Analyze sentiment of customer query
    """
    result = sentiment_analyzer(text[:512])[0]
    return {
        'sentiment': result['label'],
        'confidence': result['score']
    }

# Test sentiment analysis
test_texts = [
    "I love your product! It's amazing!",
    "I'm very frustrated with this service.",
    "How do I reset my password?"
]

print("Sentiment Analysis Results:\n")
for text in test_texts:
    sentiment = analyze_sentiment(text)
    print(f"Text: {text}")
    print(f"Sentiment: {sentiment['sentiment']} (Confidence: {sentiment['confidence']:.2%})\n")

## 9. Export and Deployment

Prepare the model and system for production deployment.

In [ ]:
# Save complete configuration
config = {
    'model_name': MODEL_NAME,
    'model_path': './fine_tuned_model',
    'vector_store_path': './vector_store',
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'retrieval_k': 3,
    'escalation_keywords': agent.escalation_keywords,
    'max_conversation_turns': 10,
    'generation_params': {
        'max_new_tokens': 200,
        'temperature': 0.7,
        'top_p': 0.9
    }
}

with open('chatbot_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Configuration saved to chatbot_config.json")

In [ ]:
# Create deployment script
deployment_script = '''#!/usr/bin/env python3
"""
Production deployment script for Customer Support Chatbot
"""

import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import gradio as gr

# Load configuration
with open('chatbot_config.json', 'r') as f:
    config = json.load(f)

# Load model and tokenizer
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(config['model_path'])
tokenizer = AutoTokenizer.from_pretrained(config['model_path'])

# Load vector store
print("Loading vector store...")
embeddings = HuggingFaceEmbeddings(model_name=config['embedding_model'])
vectorstore = FAISS.load_local(config['vector_store_path'], embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": config['retrieval_k']})

# Initialize agent
from customer_support_agent import CustomerSupportAgent
agent = CustomerSupportAgent(model, tokenizer, retriever)

# Create Gradio interface
def chat(message, history):
    result = agent.chat(message)
    return result['response']

demo = gr.ChatInterface(
    fn=chat,
    title="Customer Support Chatbot",
    description="24/7 AI-powered customer support"
)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=8080, share=False)
'''

with open('deploy.py', 'w') as f:
    f.write(deployment_script)

print("Deployment script created: deploy.py")

## 10. Summary and Next Steps

### What We've Built:
1. **Data Collection**: Sample customer support conversations and knowledge base
2. **LLM Fine-Tuning**: Fine-tuned language model using LoRA for efficient training
3. **RAG System**: Vector store and retrieval system for knowledge base access
4. **Intelligent Agent**: Chatbot with escalation logic and conversation management
5. **User Interface**: Gradio-based web interface for easy interaction
6. **Evaluation**: Comprehensive testing and performance metrics

### Next Steps for Production:
1. **Scale Data**: Collect and fine-tune on larger, real customer support datasets
2. **Improve RAG**: Add more knowledge base documents and implement hybrid search
3. **Enhanced Escalation**: Implement ML-based escalation classification
4. **Multi-turn Conversations**: Better context management across conversation turns
5. **Integration**: Connect to ticketing systems, CRM, and notification services
6. **Monitoring**: Implement logging, analytics, and continuous improvement
7. **Security**: Add authentication, rate limiting, and data privacy measures
8. **A/B Testing**: Test different models and configurations

### Files Created:
- `customer_support_conversations.json`: Training data
- `knowledge_base.json`: Knowledge base documents
- `fine_tuned_model/`: Fine-tuned model weights
- `vector_store/`: FAISS vector store
- `chatbot_config.json`: Configuration file
- `deploy.py`: Production deployment script
- `evaluation_results.png`: Evaluation visualizations

### Usage:
```python
# Initialize agent
agent = CustomerSupportAgent(model, tokenizer, retriever)

# Chat
response = agent.chat("How do I reset my password?")
print(response['response'])

# Launch UI
demo.launch()
```